<a href="https://colab.research.google.com/github/jsanchezv4/procesamiento/blob/main/fINVIZ_NEWS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#_CURSO LENGUAJE DE PROCESAMIENTO NATURAL_

Facultad de Ingeniería y Ciencias Básicas <br>
Universidad Central

JOSE MAURICIO SANCHEZ VELASQUEZ

Contenido <br>
PROBLEMATICA DEL PROYECTO <br>
OBJETIVO GENERAL<br>
OBJETIVO ESPECIFICO

#_Problemática del proyecto_

En los mercados financieros actuales, la información relevante para la toma de decisiones de inversión proviene en gran medida de fuentes institucionales como Yahoo Finance, donde se concentran noticias, análisis y eventos que impactan directamente los precios de los activos.

Sin embargo, existen limitaciones importantes:

1.  El alto volumen de noticias dificulta identificar rápidamente qué activos están siendo más relevante<br>
2. La información, aunque de calidad, no siempre está estructurada para análisis cuantitativo inmediato <br>
3. Los inversionistas pueden pasar por alto patrones en la cobertura de noticias<br>
4. No se aprovecha sistemáticamente la frecuencia de menciones como indicador de atención del mercado

Como consecuencia:

Se pierde la capacidad de detectar de forma temprana qué acciones están siendo foco del flujo informativo institucional, limitando la generación de ventajas competitivas.

#_Objetivo general_

OBJETIVO GENERAL

Desarrollar un modelo que permita analizar noticias financieras provenientes de Yahoo Finance con el fin de identificar las acciones con mayor cobertura informativa y detectar tendencias relevantes que apoyen la toma de decisiones de inversión.

#_Objetivos específicos_

1. Identificar y Recopilar Noticias financieras de yahoo Finance. Luego, se implementará un método para recopilar noticias, que puede ser a través de una API si el portal la ofrece, o mediante web scraping si la información es pública y el portal lo permite éticamente. El objetivo es obtener texto de noticias, títulos y fechas de publicación.
2. Cargar y Explorar los Datos de Noticias: Cargar los datos de noticias financieras recopilados del portal en un DataFrame de pandas.
3. Realizar una exploración inicial para entender la estructura de los datos, las columnas disponibles y los tipos de noticias. Visualizar las primeras filas y obtener estadísticas descriptivas básicas.
4. Preprocesar el Texto de las Noticias: Limpiar y normalizar el texto de las noticias para facilitar la extracción de entidades. Esto puede incluir convertir a minúsculas, eliminar puntuación, números, palabras vacías (stopwords), y aplicar lematización o stemming si es necesario.
5. Identificar Activos Financieros (Tickers): Desarrollar un método para identificar y extraer los activos financieros (acciones, empresas) mencionados en el texto de las noticias. Esto podría implicar el uso de listas de tickers y nombres de empresas, o técnicas de reconocimiento de entidades nombradas (NER) si se dispone de modelos preentrenados.
5. Analizar la Frecuencia de Cobertura: Calcular la frecuencia con la que cada activo financiero es mencionado en las noticias durante el período analizado. Identificar los activos con mayor cobertura informativa.
6. Identificar Tendencias Informativas a lo largo del Tiempo: Analizar cómo cambia la frecuencia de menciones de los activos principales a lo largo del tiempo.
7. Detectar incrementos inusuales en la atención mediática hacia ciertos activos, lo que podría indicar eventos relevantes o cambios en el interés del mercado.
8. Visualizar Frecuencia y Tendencias: Crear gráficos y visualizaciones para presentar los resultados del análisis de frecuencia y tendencias. Esto puede incluir gráficos de barras para los activos más mencionados, gráficos de líneas para mostrar la evolución de las menciones a lo largo del tiempo, y cualquier otra visualización que ayude a comprender los patrones.
9. Generar Insights para la Toma de Decisiones de Inversión: Interpretar los resultados obtenidos. Identificar qué acciones están recibiendo mayor atención, si hay tendencias emergentes, y cómo esta información puede complementar el análisis del portafolio del fondo. Proporcionar un resumen de los hallazgos clave.
10. Final Task: Proporcionar un resumen final del análisis de noticias del portal financiero y sus implicaciones para las decisiones de inversión.

### Paso 1: Obtener las noticias de Finviz

Primero, necesitamos instalar las bibliotecas `requests` para hacer solicitudes HTTP y `BeautifulSoup` para analizar el contenido HTML de la página web de Finviz.

In [1]:
pip install requests beautifulsoup4

Ahora, utilizaremos estas bibliotecas para acceder a la página de noticias de Finviz y extraer los titulares. Es importante incluir un `User-Agent` en los encabezados de la solicitud para que la página web nos identifique como un navegador web estándar y evitar ser bloqueados.

In [2]:
import requests
from bs4 import BeautifulSoup

# URL de las noticias de Finviz. Puedes ajustar esta URL si quieres un tipo de noticia específico.
FINVIZ_NEWS_URL = 'https://finviz.com/news.ashx'

# Un encabezado User-Agent para simular una solicitud desde un navegador web
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

try:
    response = requests.get(FINVIZ_NEWS_URL, headers=headers)
    response.raise_for_status() # Lanza una excepción para códigos de estado de error HTTP

    soup = BeautifulSoup(response.text, 'html.parser')

    # En Finviz, los enlaces de noticias a menudo tienen el atributo target="_blank".
    # Buscamos todos los enlaces (<a>) que tienen este atributo.
    news_items_found = soup.find_all('a', target='_blank')

    # Filtramos para asegurarnos de que sean enlaces de noticias y no otros elementos.
    # A menudo, los enlaces de noticias están dentro de un div o td con una clase específica, o simplemente
    # tienen un texto descriptivo y un href completo.
    # Por ahora, nos quedaremos con los que tienen target="_blank" y un texto no vacío.
    filtered_news_items = []
    for item in news_items_found:
        if item.get('href') and item.get_text(strip=True): # Asegurarse de que el enlace tiene href y texto
            filtered_news_items.append(item)

    print(f"Se encontraron {len(filtered_news_items)} titulares de noticias.\n")

    # Imprime los primeros 10 titulares para verificar
    print("Primeros 10 titulares:\n")
    for i, item in enumerate(filtered_news_items[:10]):
        title = item.get_text(strip=True) # Usar strip=True para eliminar espacios en blanco al inicio/final
        link = item['href']
        print(f"Título: {title}\nEnlace: {link}\n")

except requests.exceptions.RequestException as e:
    print(f"Error al obtener las noticias: {e}")
except Exception as e:
    print(f"Ocurrió un error: {e}")

Se encontraron 182 titulares de noticias.

Primeros 10 titulares:

Título: Head of Harvard’s Endowment Tells Board He Plans to Retire
Enlace: https://www.wsj.com/finance/investing/head-of-harvards-endowment-tells-board-he-plans-to-retire-001f2766?mod=rss_markets_main

Título: Bond Futures at Risk From Rapid Hedging Overhaul as Yields Climb
Enlace: https://www.bloomberg.com/news/articles/2026-05-15/bond-futures-at-risk-from-rapid-hedging-overhaul-as-yields-climb

Título: Yield-Hungry Investors Bet on Credit as Government Debt Sours
Enlace: https://www.bloomberg.com/news/articles/2026-05-15/inflation-risk-gives-corporate-bonds-the-edge-over-sovereigns

Título: Curious Yen Spikes Have Traders Gaming Japan ‘Warning Shots’
Enlace: https://www.bloomberg.com/news/articles/2026-05-15/curious-yen-spikes-have-traders-gaming-out-japan-warning-shots

Título: NextEra Energy Is in Talks to Combine With Dominion Energy: FT
Enlace: https://www.bloomberg.com/news/articles/2026-05-16/nextera-energy-is-i

### Paso 2.1: Cargar la lista de tickers del S&P 500 para validación

In [3]:
import pandas as pd
import requests

SP500_WIKIPEDIA_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers_wiki = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

try:
    response_wiki = requests.get(SP500_WIKIPEDIA_URL, headers=headers_wiki)
    response_wiki.raise_for_status() # Lanza una excepción para códigos de estado de error HTTP
    sp500_table = pd.read_html(response_wiki.text)[0] # La primera tabla suele ser la de tickers

    # Extraer tickers y convertir a un conjunto para una búsqueda eficiente
    sp500_tickers = set(sp500_table['Symbol'].tolist())

    # Eliminar el ticker ambiguo de una sola letra 'A' (Agilent Technologies)
    # ya que a menudo crea falsos positivos con palabras comunes
    if 'A' in sp500_tickers:
        sp500_tickers.remove('A')

    print(f"Se cargaron {len(sp500_tickers)} tickers del S&P 500 de Wikipedia (excluyendo 'A').")
except requests.exceptions.RequestException as e:
    print(f"Error al obtener la lista del S&P 500 de Wikipedia: {e}")
    sp500_tickers = set() # Asegurarse de que sea un conjunto vacío si la carga falla
except Exception as e:
    print(f"Ocurrió un error al procesar la tabla del S&P 500: {e}")
    sp500_tickers = set() # Asegurarse de que sea un conjunto vacío si el procesamiento falla

Se cargaron 502 tickers del S&P 500 de Wikipedia (excluyendo 'A').


/tmp/ipykernel_2804/2833525074.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500_table = pd.read_html(response_wiki.text)[0] # La primera tabla suele ser la de tickers


### Paso 4: Validar tickers extraídos contra la lista del S&P 500 y refinar los falsos positivos

In [20]:
# Filtrar news_with_sentiment_and_tickers para incluir solo aquellos con tickers S&P 500 validados
news_with_sentiment_and_sp500_tickers = []
all_extracted_tickers_from_content = set() # Para recopilar todos los tickers encontrados en el contenido

for news_item in news_with_sentiment_and_tickers:
    validated_tickers = []
    # Recopilar todos los tickers encontrados (antes de validar)
    all_extracted_tickers_from_content.update(news_item['potential_tickers'])

    for ticker in news_item['potential_tickers']:
        if ticker in sp500_tickers:
            validated_tickers.append(ticker)

    if validated_tickers:
        # Crear una copia de news_item y actualizar 'potential_tickers'
        item_copy = news_item.copy()
        item_copy['potential_tickers'] = sorted(list(set(validated_tickers))) # Eliminar duplicados y ordenar
        news_with_sentiment_and_sp500_tickers.append(item_copy)

print(f"Después de la validación del S&P 500, se encontraron {len(news_with_sentiment_and_sp500_tickers)} noticias con tickers válidos del S&P 500.\n")

# Identificar tickers falsos positivos más actualizados
# Definir algunos términos comunes que son propensos a ser falsos positivos y no son tickers
known_non_tickers = {
    'US', 'AI', 'BOJ', 'IPO', 'FOMC', 'GDP', 'AM', 'CEO', 'CFO', 'ETF', 'NY', 'NJ', 'UK', 'CPU', 'GPU', 'S', 'P', 'I', 'X',
    'NASDAQ', 'NYSE', 'MSCI', 'ADR', 'SEC', 'FASB', 'GAAP', 'IRS', 'DOJ', 'FTC', 'FED', 'ECB', 'RBI', 'BOC', 'BOE', 'PBOC',
    'CSRC', 'MAS', 'AMEX', 'LSE', 'TSX', 'ASX', 'SIX', 'BSE', 'NSE', 'KOSPI', 'TOPIX', 'NIKKEI', 'HSI', 'DAX', 'CAC', 'SMI',
    'IBEX', 'FTSE', 'DOW', 'ARKK', 'SPX', 'QQQ', 'DIA', 'VIX', 'GOLD', 'OIL', 'GAS', 'EUR', 'USD', 'GBP', 'JPY', 'CAD', 'AUD',
    'NZD', 'CHF', 'CNY', 'INR', 'BRL', 'MXN', 'KRW', 'HKD', 'SGD', 'CLP', 'COP', 'PEN', 'ARS', 'ZAR', 'RUB', 'TRY', 'PLN',
    'SEK', 'NOK', 'DKK', 'HUF', 'CZK', 'ILS', 'SGD', 'THB', 'MYR', 'IDR', 'PHP', 'VND', 'AED', 'SAR', 'QAR', 'KWD', 'BHD',
    'OMR', 'EGP', 'NGN', 'GHS', 'KES', 'UGX', 'TZS', 'ZMW', 'XAU', 'XAG', 'XPT', 'XPD', 'XBT', 'ETH', 'XRP', 'LTC', 'BCH',
    'ADA', 'DOT', 'UNI', 'LINK', 'SOL', 'MATIC', 'XLM', 'VET', 'EOS', 'TRX', 'NEO', 'DASH', 'XMR', 'ETC', 'ZEC', 'FIL', 'ICP',
    'GRT', 'MKR', 'AAVE', 'COMP', 'SNX', 'YFI', 'UMA', 'CRV', 'SUSHI', 'BADGER', 'FORTH', 'AMP', 'CHZ', 'ENJ', 'MANA', 'SAND',
    'AXS', 'THETA', 'XTZ', 'EGLD', 'ALGO', 'WAVES', 'DOGE', 'SHIB', 'PEPE', 'FLOKI', 'LEASH', 'BONE', 'FEG', 'SAFEMOON', 'DNT',
    'CVC', 'OMG', 'GOLEM', 'LRC', 'KNC', 'BAL', 'REN', 'OCEAN', 'RLC', 'NMR', 'ANT', 'BAND', 'DIA', 'STX', 'CELO', 'NEAR',
    'AVAX', 'FTM', 'ONE', 'ICX', 'IOTX', 'RVN', 'SC', 'ZIL', 'OMG', 'DGB', 'BTG', 'DCR', 'LSK', 'WTC', 'GAS', 'ARK', 'FCT',
    'MAID', 'FUN', 'POWR', 'SALT', 'KMD', 'RDD', 'VERI', 'REP', 'GBYTE', 'NXT', 'SYS', 'PPC', 'EMC', 'NLG', 'NAV', 'ARK',
    'EXP', 'LKK', 'VIB', 'MCO', 'REQ', 'ENG', 'MTL', 'AGI', 'RCN', 'KIN', 'SUB', 'STORJ', 'ADX', 'COB', 'CTR', 'SNGLS', 'PRO',
    'DNT', 'POE', 'QSP', 'AMB', 'VEE', 'OST', 'CDT', 'DLT', 'EDG', 'LINK', 'WAN', 'POLY', 'QLC', 'PAI', 'MTH', 'DBC', 'RNT',
    'HPB', 'ZPT', 'WPR', 'SWFTC', 'ITC', 'ELA', 'ELEC', 'GET', 'APPC', 'QKC', 'ABYSS', 'GTO', 'INS', 'LEND', 'DGD', 'EOS',
    'ADA', 'TRX', 'MIOTA', 'DASH', 'XMR', 'ZEC', 'XTZ', 'VET', 'NEO', 'ETC', 'BNB', 'XRP', 'LTC', 'BCH', 'XLM', 'DOGE',
    'TRON', 'DOT', 'LINK', 'UNI', 'ADA', 'SOL', 'LUNA', 'AVAX', 'MATIC', 'ICP', 'WAVES', 'FIL', 'VET', 'THETA', 'EOS', 'MKR',
    'AAVE', 'COMP', 'SNX', 'YFI', 'UMA', 'CRV', 'SUSHI', 'GRT', 'ICX', 'ALGO', 'ZIL', 'DGB', 'QTUM', 'IOST', 'RVN', 'OMG',
    'WAN', 'ONT', 'NANO', 'LSK', 'DCR', 'BTG', 'KMD', 'ENJ', 'MANA', 'SAND', 'AXS', 'CHZ', 'EGLD', 'ONE', 'ZRX', 'BAT',
    'DNT', 'CVC', 'GOLEM', 'LRC', 'KNC', 'BAL', 'REN', 'OCEAN', 'RLC', 'NMR', 'ANT', 'BAND', 'DIA', 'STX', 'CELO', 'NEAR',
    'AVAX', 'FTM', 'ONE', 'ICX', 'IOTX', 'RVN', 'SC', 'ZIL', 'OMG', 'DGB', 'BTG', 'DCR', 'LSK', 'WTC', 'GAS', 'ARK', 'FCT',
    'MAID', 'FUN', 'POWR', 'SALT', 'KMD', 'RDD', 'VERI', 'REP', 'GBYTE', 'NXT', 'SYS', 'PPC', 'EMC', 'NLG', 'NAV', 'ARK',
    'EXP', 'LKK', 'VIB', 'MCO', 'REQ', 'ENG', 'MTL', 'AGI', 'RCN', 'KIN', 'SUB', 'STORJ', 'ADX', 'COB', 'CTR', 'SNGLS', 'PRO',
    'DNT', 'POE', 'QSP', 'AMB', 'VEE', 'OST', 'CDT', 'DLT', 'EDG', 'LINK', 'WAN', 'POLY', 'QLC', 'PAI', 'MTH', 'DBC', 'RNT',
    'HPB', 'ZPT', 'WPR', 'SWFTC', 'ITC', 'ELA', 'ELEC', 'GET', 'APPC', 'QKC', 'ABYSS', 'GTO', 'INS', 'LEND', 'DGD', 'ETH',
    'BTC',
}

false_positive_tickers = []
for ticker in all_extracted_tickers_from_content:
    if ticker not in sp500_tickers and ticker in known_non_tickers:
        false_positive_tickers.append(ticker)

# Filtrar tickers de 1-3 letras que no son S&P 500 y no están en known_non_tickers pero son probablemente acrónimos
# Excluir 'A' ya que se maneja explícitamente y puede ser un falso positivo solo si es una palabra.
for ticker in all_extracted_tickers_from_content:
    if 1 <= len(ticker) <= 3 and ticker.isalpha() and ticker not in sp500_tickers and ticker not in known_non_tickers and ticker != 'A':
        false_positive_tickers.append(ticker)

# Asegurarse de que 'A' si fue re-introducido en all_extracted_tickers_from_content y no está en S&P 500 sea un false positive
if 'A' in all_extracted_tickers_from_content and 'A' not in sp500_tickers: # sp500_tickers no tiene 'A' intencionalmente
    if 'A' not in false_positive_tickers:
        false_positive_tickers.append('A')

false_positive_tickers = sorted(list(set(false_positive_tickers)))

print(f"Se identificaron {len(false_positive_tickers)} posibles tickers falsos positivos (actualizados con contenido de artículos):\n{false_positive_tickers}\n")

Después de la validación del S&P 500, se encontraron 2 noticias con tickers válidos del S&P 500.

Se identificaron 29 posibles tickers falsos positivos (actualizados con contenido de artículos):
['A', 'AI', 'AM', 'CBD', 'CEO', 'CPI', 'ETF', 'FDA', 'FT', 'G', 'GDP', 'GFL', 'I', 'II', 'INO', 'IPO', 'LNG', 'NANO', 'NFL', 'NPR', 'NYC', 'P', 'PPI', 'S', 'U', 'UAE', 'UK', 'US', 'X']



In [18]:
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Descargar el léxico de VADER si no está disponible (para asegurar que la inicialización funcione)
nltk.download('vader_lexicon', quiet=True)

# Inicializar el analizador de sentimiento VADER
analyzer = SentimentIntensityAnalyzer()

news_with_sentiment_and_tickers = []

for item in filtered_news_items:
    title = item.get_text(strip=True)
    link = item['href']

    # Extraer posibles tickers (palabras en mayúsculas) del título.
    # Utilizamos \b para asegurar que sea una palabra completa y buscamos de 1 a 5 caracteres.
    # La validación contra la lista del S&P 500 y los falsos positivos se realizará en celdas posteriores.
    potential_tickers = re.findall(r'\b[A-Z]{1,5}\b', title)

    # Realizar análisis de sentimiento en el título de la noticia
    sentiment_scores = analyzer.polarity_scores(title)

    news_with_sentiment_and_tickers.append({
        'title': title,
        'link': link,
        'potential_tickers': potential_tickers,
        'sentiment': sentiment_scores
    })

print(f"Se procesaron {len(news_with_sentiment_and_tickers)} noticias con extracción de tickers potenciales y análisis de sentimiento.")

# Mostrar los primeros 5 elementos procesados para verificación
print("\nPrimeros 5 elementos procesados:\n")
for i, news in enumerate(news_with_sentiment_and_tickers[:5]):
    print(f"News {i+1}:")
    print(f"  Title: {news['title']}")
    print(f"  Link: {news['link']}")
    print(f"  Potential Tickers: {news['potential_tickers']}")
    print(f"  Sentiment: {news['sentiment']}\n")

Se procesaron 182 noticias con extracción de tickers potenciales y análisis de sentimiento.

Primeros 5 elementos procesados:

News 1:
  Title: Head of Harvard’s Endowment Tells Board He Plans to Retire
  Link: https://www.wsj.com/finance/investing/head-of-harvards-endowment-tells-board-he-plans-to-retire-001f2766?mod=rss_markets_main
  Potential Tickers: []
  Sentiment: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}

News 2:
  Title: Bond Futures at Risk From Rapid Hedging Overhaul as Yields Climb
  Link: https://www.bloomberg.com/news/articles/2026-05-15/bond-futures-at-risk-from-rapid-hedging-overhaul-as-yields-climb
  Potential Tickers: []
  Sentiment: {'neg': 0.174, 'neu': 0.826, 'pos': 0.0, 'compound': -0.2732}

News 3:
  Title: Yield-Hungry Investors Bet on Credit as Government Debt Sours
  Link: https://www.bloomberg.com/news/articles/2026-05-15/inflation-risk-gives-corporate-bonds-the-edge-over-sovereigns
  Potential Tickers: []
  Sentiment: {'neg': 0.207, 'neu': 0.579,

In [21]:
sp500_validated_tickers_in_news = set()
for news_item in news_with_sentiment_and_sp500_tickers:
    for ticker in news_item['potential_tickers']:
        sp500_validated_tickers_in_news.add(ticker)

print(f"\nTickers definitivos encontrados en las noticias (validados con S&P 500):\n{sorted(list(sp500_validated_tickers_in_news))}")
print(f"\nTickers identificados como falsos positivos (actualizados con contenido de artículos):\n{false_positive_tickers}\n")


Tickers definitivos encontrados en las noticias (validados con S&P 500):
['MA']

Tickers identificados como falsos positivos (actualizados con contenido de artículos):
['A', 'AI', 'AM', 'CBD', 'CEO', 'CPI', 'ETF', 'FDA', 'FT', 'G', 'GDP', 'GFL', 'I', 'II', 'INO', 'IPO', 'LNG', 'NANO', 'NFL', 'NPR', 'NYC', 'P', 'PPI', 'S', 'U', 'UAE', 'UK', 'US', 'X']



In [22]:
total_compound = 0
total_pos = 0
total_neg = 0
total_neu = 0
news_count_with_sp500_tickers = 0

for news_item in news_with_sentiment_and_sp500_tickers:
    if news_item['potential_tickers']: # Solo considerar noticias con tickers S&P 500 validados
        sentiment = news_item['sentiment']
        total_compound += sentiment['compound']
        total_pos += sentiment['pos']
        total_neg += sentiment['neg']
        total_neu += sentiment['neu']
        news_count_with_sp500_tickers += 1

if news_count_with_sp500_tickers > 0:
    avg_compound = total_compound / news_count_with_sp500_tickers
    avg_pos = total_pos / news_count_with_sp500_tickers
    avg_neg = total_neg / news_count_with_sp500_tickers
    avg_neu = total_neu / news_count_with_sp500_tickers

    print(f"\n--- Sentimiento Promedio para noticias con tickers S&P 500 validados ({news_count_with_sp500_tickers} noticias) ---")
    print(f"  Compuesto Promedio: {avg_compound:.4f}")
    print(f"  Positivo Promedio: {avg_pos:.4f}")
    print(f"  Negativo Promedio: {avg_neg:.4f}")
    print(f"  Neutro Promedio: {avg_neu:.4f}")
else:
    print("No se encontraron noticias con tickers S&P 500 validados para calcular el sentimiento promedio.")


--- Sentimiento Promedio para noticias con tickers S&P 500 validados (2 noticias) ---
  Compuesto Promedio: 0.0129
  Positivo Promedio: 0.1120
  Negativo Promedio: 0.1075
  Neutro Promedio: 0.7805


In [24]:
import pandas as pd

def summarize_ticker_sentiment(ticker, news_list):
    """
    Calcula el sentimiento promedio y recopila noticias para un ticker específico.
    """
    relevant_news = []
    total_compound = 0
    total_pos = 0
    total_neg = 0
    total_neu = 0

    for news_item in news_list:
        # Verificar si el ticker está en la lista de tickers potenciales validados de la noticia
        if ticker in news_item.get('potential_tickers', []):
            relevant_news.append(news_item)
            sentiment = news_item['sentiment']
            total_compound += sentiment['compound']
            total_pos += sentiment['pos']
            total_neg += sentiment['neg']
            total_neu += sentiment['neu']

    number_of_news_articles = len(relevant_news)

    if number_of_news_articles > 0:
        return {
            'ticker': ticker,
            'number_of_news_articles': number_of_news_articles,
            'average_sentiment': {
                'compound': total_compound / number_of_news_articles,
                'positive': total_pos / number_of_news_articles,
                'negative': total_neg / number_of_news_articles,
                'neutral': total_neu / number_of_news_articles
            },
            'relevant_news': relevant_news
        }
    else:
        return {'ticker': ticker, 'number_of_news_articles': 0, 'message': f"No se encontraron noticias para el ticker {ticker}."}

ticker_sentiment_summary = []

for ticker in sorted(list(sp500_validated_tickers_in_news)):
    summary = summarize_ticker_sentiment(ticker, news_with_sentiment_and_sp500_tickers)
    if 'average_sentiment' in summary:
        ticker_sentiment_summary.append({
            'Ticker': summary['ticker'],
            'Número de Noticias': summary['number_of_news_articles'],
            'Sentimiento Compuesto Promedio': f"{summary['average_sentiment']['compound']:.4f}",
            'Sentimiento Positivo Promedio': f"{summary['average_sentiment']['positive']:.4f}",
            'Sentimiento Negativo Promedio': f"{summary['average_sentiment']['negative']:.4f}",
            'Sentimiento Neutro Promedio': f"{summary['average_sentiment']['neutral']:.4f}"
        })

# Crear un DataFrame de pandas para una mejor visualización
df_sentiment_summary = pd.DataFrame(ticker_sentiment_summary)

# Ordenar por el número de noticias para ver los tickers más mencionados primero
df_sentiment_summary = df_sentiment_summary.sort_values(by='Número de Noticias', ascending=False).reset_index(drop=True)

display(df_sentiment_summary)


,Ticker,Número de Noticias,Sentimiento Compuesto Promedio,Sentimiento Positivo Promedio,Sentimiento Negativo Promedio,Sentimiento Neutro Promedio
0,MA,2,0.0129,0.1120,0.1075,0.7805


In [25]:
# Re-ejecutar la celda que filtra y valida los tickers con la lista actualizada de known_non_tickers
# para que 'FOX' sea excluido correctamente si está en news_with_sentiment_and_tickers

# Filtrar news_with_sentiment_and_tickers para incluir solo aquellos con tickers S&P 500 validados
news_with_sentiment_and_sp500_tickers = []
all_extracted_tickers_from_content = set() # Para recopilar todos los tickers encontrados en el contenido

for news_item in news_with_sentiment_and_tickers:
    validated_tickers = []
    # Recopilar todos los tickers encontrados (antes de validar)
    all_extracted_tickers_from_content.update(news_item['potential_tickers'])

    for ticker in news_item['potential_tickers']:
        if ticker in sp500_tickers:
            validated_tickers.append(ticker)

    # Aquí se aplica el filtro de known_non_tickers (donde FOX debería estar ahora)
    validated_tickers = [t for t in validated_tickers if t not in known_non_tickers]

    if validated_tickers:
        # Crear una copia de news_item y actualizar 'potential_tickers'
        item_copy = news_item.copy()
        item_copy['potential_tickers'] = sorted(list(set(validated_tickers))) # Eliminar duplicados y ordenar
        news_with_sentiment_and_sp500_tickers.append(item_copy)

print(f"Después de la validación del S&P 500, se encontraron {len(news_with_sentiment_and_sp500_tickers)} noticias con tickers válidos del S&P 500.\n")

# Re-identificar tickers falsos positivos para que 'FOX' aparezca si se extrajo y no es S&P 500
false_positive_tickers = []
for ticker in all_extracted_tickers_from_content:
    if ticker not in sp500_tickers and ticker in known_non_tickers:
        false_positive_tickers.append(ticker)

# Filtrar tickers de 1-3 letras que no son S&P 500 y no están en known_non_tickers pero son probablemente acrónimos
# Excluir 'A' ya que se maneja explícitamente y puede ser un falso positivo solo si es una palabra.
for ticker in all_extracted_tickers_from_content:
    if 1 <= len(ticker) <= 3 and ticker.isalpha() and ticker not in sp500_tickers and ticker not in known_non_tickers and ticker != 'A':
        false_positive_tickers.append(ticker)

# Asegurarse de que 'A' si fue re-introducido en all_extracted_tickers_from_content y no está en S&P 500 sea un false positive
if 'A' in all_extracted_tickers_from_content and 'A' not in sp500_tickers: # sp500_tickers no tiene 'A' intencionalmente
    if 'A' not in false_positive_tickers:
        false_positive_tickers.append('A')

false_positive_tickers = sorted(list(set(false_positive_tickers)))

print(f"Se identificaron {len(false_positive_tickers)} posibles tickers falsos positivos (actualizados con contenido de artículos):\n{false_positive_tickers}\n")

Después de la validación del S&P 500, se encontraron 2 noticias con tickers válidos del S&P 500.

Se identificaron 29 posibles tickers falsos positivos (actualizados con contenido de artículos):
['A', 'AI', 'AM', 'CBD', 'CEO', 'CPI', 'ETF', 'FDA', 'FT', 'G', 'GDP', 'GFL', 'I', 'II', 'INO', 'IPO', 'LNG', 'NANO', 'NFL', 'NPR', 'NYC', 'P', 'PPI', 'S', 'U', 'UAE', 'UK', 'US', 'X']



In [26]:
sp500_validated_tickers_in_news = set()
for news_item in news_with_sentiment_and_sp500_tickers:
    for ticker in news_item['potential_tickers']:
        sp500_validated_tickers_in_news.add(ticker)

print(f"\nTickers definitivos encontrados en las noticias (validados con S&P 500):\n{sorted(list(sp500_validated_tickers_in_news))}")
print(f"\nTickers identificados como falsos positivos (actualizados con contenido de artículos):\n{false_positive_tickers}\n")


Tickers definitivos encontrados en las noticias (validados con S&P 500):
['MA']

Tickers identificados como falsos positivos (actualizados con contenido de artículos):
['A', 'AI', 'AM', 'CBD', 'CEO', 'CPI', 'ETF', 'FDA', 'FT', 'G', 'GDP', 'GFL', 'I', 'II', 'INO', 'IPO', 'LNG', 'NANO', 'NFL', 'NPR', 'NYC', 'P', 'PPI', 'S', 'U', 'UAE', 'UK', 'US', 'X']



In [27]:
total_compound = 0
total_pos = 0
total_neg = 0
total_neu = 0
news_count_with_sp500_tickers = 0

for news_item in news_with_sentiment_and_sp500_tickers:
    if news_item['potential_tickers']: # Solo considerar noticias con tickers S&P 500 validados
        sentiment = news_item['sentiment']
        total_compound += sentiment['compound']
        total_pos += sentiment['pos']
        total_neg += sentiment['neg']
        total_neu += sentiment['neu']
        news_count_with_sp500_tickers += 1

if news_count_with_sp500_tickers > 0:
    avg_compound = total_compound / news_count_with_sp500_tickers
    avg_pos = total_pos / news_count_with_sp500_tickers
    avg_neg = total_neg / news_count_with_sp500_tickers
    avg_neu = total_neu / news_count_with_sp500_tickers

    print(f"\n--- Sentimiento Promedio para noticias con tickers S&P 500 validados ({news_count_with_sp500_tickers} noticias) ---")
    print(f"  Compuesto Promedio: {avg_compound:.4f}")
    print(f"  Positivo Promedio: {avg_pos:.4f}")
    print(f"  Negativo Promedio: {avg_neg:.4f}")
    print(f"  Neutro Promedio: {avg_neu:.4f}")
else:
    print("No se encontraron noticias con tickers S&P 500 validados para calcular el sentimiento promedio.")


--- Sentimiento Promedio para noticias con tickers S&P 500 validados (2 noticias) ---
  Compuesto Promedio: 0.0129
  Positivo Promedio: 0.1120
  Negativo Promedio: 0.1075
  Neutro Promedio: 0.7805


In [28]:
import pandas as pd

ticker_sentiment_summary = []

for ticker in sorted(list(sp500_validated_tickers_in_news)):
    summary = summarize_ticker_sentiment(ticker, news_with_sentiment_and_sp500_tickers)
    if 'average_sentiment' in summary:
        ticker_sentiment_summary.append({
            'Ticker': summary['ticker'],
            'Número de Noticias': summary['number_of_news_articles'],
            'Sentimiento Compuesto Promedio': f"{summary['average_sentiment']['compound']:.4f}",
            'Sentimiento Positivo Promedio': f"{summary['average_sentiment']['positive']:.4f}",
            'Sentimiento Negativo Promedio': f"{summary['average_sentiment']['negative']:.4f}",
            'Sentimiento Neutro Promedio': f"{summary['average_sentiment']['neutral']:.4f}"
        })

# Crear un DataFrame de pandas para una mejor visualización
df_sentiment_summary = pd.DataFrame(ticker_sentiment_summary)

# Ordenar por el número de noticias para ver los tickers más mencionados primero
df_sentiment_summary = df_sentiment_summary.sort_values(by='Número de Noticias', ascending=False).reset_index(drop=True)

display(df_sentiment_summary)

,Ticker,Número de Noticias,Sentimiento Compuesto Promedio,Sentimiento Positivo Promedio,Sentimiento Negativo Promedio,Sentimiento Neutro Promedio
0,MA,2,0.0129,0.1120,0.1075,0.7805


### Paso 5.2: Mostrar Noticias Detalladas para un Ticker Específico

In [29]:
# Elige el ticker S&P 500 validado que quieres analizar en detalle
chosen_ticker = 'AMZN' # @param {type:"string"}

# Obtener el resumen detallado para el ticker elegido
detail_summary = summarize_ticker_sentiment(chosen_ticker, news_with_sentiment_and_sp500_tickers)

if 'message' in detail_summary:
    print(detail_summary['message'])
else:
    print(f"\n--- Noticias Detalladas para el Ticker {detail_summary['ticker']} ({detail_summary['number_of_news_articles']} noticias) ---")
    print(f"  Sentimiento Compuesto Promedio: {detail_summary['average_sentiment']['compound']:.4f}")
    print(f"  Sentimiento Positivo Promedio: {detail_summary['average_sentiment']['positive']:.4f}")
    print(f"  Sentimiento Negativo Promedio: {detail_summary['average_sentiment']['negative']:.4f}")
    print(f"  Sentimiento Neutro Promedio: {detail_summary['average_sentiment']['neutral']:.4f}\n")

    print("Noticias relevantes:\n")
    for news in detail_summary['relevant_news']:
        print(f"  - Título: {news['title']}")
        print(f"    Enlace: {news['link']}")
        print(f"    Sentimiento (compuesto): {news['sentiment']['compound']:.4f}\n")

No se encontraron noticias para el ticker AMZN.
